# Assigned concepts vs billed movements
### Conceptos asignados vs movimientos facturados

**Executed public case study · completely synthetic · period 2030-09**

Billing preparation does not guarantee that every line was posted at the expected amount. This control compares both quantities and value, detects missing billing and unexpected movements, and separates explainable adjustments from unresolved exposure.

Preparar la facturación no garantiza que cada línea haya sido facturada con el importe esperado. Este control compara cantidades y valor, detecta faltantes y movimientos inesperados, y separa ajustes explicables de exposición pendiente.

[Case study / caso](../docs/cases/02_billing.md) · [Español](../docs/cases/02_billing.es.md)

## 1. Universe and contracts / Universo y contratos

assigned.csv (expected lines/amounts) and billed.csv (actual posted lines/amounts, with a fictitious manual_adjustment flag). One row represents one financial line, not a student balance.

The comparison grain is **student + program after selecting period**. Every event must have a unique row ID. Currency values are fictional. No Oracle connection or environment secret is read.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "data" / "synthetic").is_dir())
sys.path.insert(0, str(root / "src"))
from financial_controls.demo import load_inputs
from financial_controls.contracts import scope, unique, KEYS
from financial_controls.money import money_column, format_cents
from financial_controls.reconciliation import reconcile, summarize
from financial_controls.controls import enrollment_control, billing_control, scholarship_control, units
from financial_controls.reporting import summarize_review, display_detail

period = "2030-09"
data = load_inputs(root)

In [2]:
manifest = pd.DataFrame([
    {"source": name, "rows": len(data[name]), "columns": len(data[name].columns)}
    for name in ["assigned","billed"]
])
display(manifest)
print("assigned")
display(data["assigned"])

print("billed")
display(data["billed"])

,source,rows,columns
0,assigned,9,6
1,billed,9,6


assigned


,row_id,student_ref,program_ref,period,amount,prior_period
0,SYN-A01,SYN-S001,SYN-A,2030-09,100.00,N
1,SYN-A02,SYN-S001,SYN-A,2030-09,100.00,N
2,SYN-A03,SYN-S001,SYN-B,2030-09,50.00,N
3,SYN-A04,SYN-S002,SYN-A,2030-09,100.00,N
4,SYN-A05,SYN-S003,SYN-B,2030-09,150.00,N
5,SYN-A06,SYN-S003,SYN-B,2030-09,150.00,N
6,SYN-A07,SYN-S004,SYN-A,2030-09,100.00,N
7,SYN-A08,SYN-S004,SYN-A,2030-09,100.00,Y
8,SYN-A09,SYN-S006,SYN-B,2030-09,80.00,N


billed


,row_id,student_ref,program_ref,period,amount,manual_adjustment
0,SYN-B01,SYN-S001,SYN-A,2030-09,100.00,N
1,SYN-B02,SYN-S001,SYN-A,2030-09,100.00,N
2,SYN-B03,SYN-S001,SYN-B,2030-09,50.00,N
3,SYN-B04,SYN-S002,SYN-A,2030-09,100.00,N
4,SYN-B05,SYN-S003,SYN-B,2030-09,150.00,N
5,SYN-B06,SYN-S004,SYN-A,2030-09,100.00,N
6,SYN-B07,SYN-S004,SYN-A,2030-09,100.00,N
7,SYN-B08,SYN-S004,SYN-A,2030-09,100.00,Y
8,SYN-B09,SYN-S007,SYN-A,2030-09,90.00,Y


## 2. Normalize and aggregate / Normalizar y agregar

Keys stay strings; null keys are not converted to zero. Money is parsed from decimal strings and becomes integer cents before summing. Cardinality is declared after grouping.

In [3]:
assigned = scope(data["assigned"], period, ["amount"])
billed = scope(data["billed"], period, ["amount", "manual_adjustment"])
conversion = assigned[["row_id", "amount"]].copy()
conversion["integer_cents"] = money_column(conversion["amount"])
display(conversion)
expected = summarize(units(assigned, money=True))
actual = summarize(units(billed, money=True))
preview = expected.merge(actual, on=KEYS, how="outer", validate="one_to_one",
                         suffixes=("_expected", "_actual"), indicator=True)
display(preview)

,row_id,amount,integer_cents
0,SYN-A01,100.00,10000
1,SYN-A02,100.00,10000
2,SYN-A03,50.00,5000
3,SYN-A04,100.00,10000
4,SYN-A05,150.00,15000
5,SYN-A06,150.00,15000
6,SYN-A07,100.00,10000
7,SYN-A08,100.00,10000
8,SYN-A09,80.00,8000


,student_ref,program_ref,lines_expected,cents_expected,lines_actual,cents_actual,_merge
0,SYN-S001,SYN-A,2.0,20000.0,2.0,20000.0,both
1,SYN-S001,SYN-B,1.0,5000.0,1.0,5000.0,both
2,SYN-S002,SYN-A,1.0,10000.0,1.0,10000.0,both
3,SYN-S003,SYN-B,2.0,30000.0,1.0,15000.0,both
4,SYN-S004,SYN-A,2.0,20000.0,3.0,30000.0,both
5,SYN-S006,SYN-B,1.0,8000.0,NaN,NaN,left_only
6,SYN-S007,SYN-A,NaN,NaN,1.0,9000.0,right_only


## 3. A contract must fail when violated / Un contrato debe fallar

A duplicate event or rule must not multiply rows silently. The following deliberate synthetic error is caught and reported by category only.

In [4]:
bad_input = pd.concat([data["assigned"], data["assigned"].iloc[[0]]], ignore_index=True)
try:
    scope(bad_input, period, ["amount"])
except ValueError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("Duplicate event should fail")

ValueError: Duplicate key violates cardinality contract


## 4. Compare, classify and explain / Comparar, clasificar y justificar

Only synthetic manual-adjustment rows explain excess actual billing. Line and amount capacities are capped independently; closing one dimension does not close the other. Missing billed amounts remain pending. Negative adjustment capacity is rejected for explicit review.

Sólo las filas sintéticas de ajuste manual explican excesos facturados. Las capacidades de cantidad e importe se limitan por separado; cerrar una dimensión no cierra la otra. Los faltantes quedan pendientes. Una capacidad de ajuste negativa requiere revisión explícita.

Difference = actual − expected. Raw status and review status are separate.

In [5]:
result = billing_control(data["assigned"], data["billed"], period=period)
display(display_detail(result))
display(summarize_review(result))

,student_ref,program_ref,lines_expected,lines_actual,amount_expected,amount_actual,difference_lines,difference_amount,justified_amount,pending_amount,raw_status,review_status,presence
0,SYN-S001,SYN-A,2,2,200.00,200.00,0,0.00,0.00,0.00,OK,OK,both
1,SYN-S001,SYN-B,1,1,50.00,50.00,0,0.00,0.00,0.00,OK,OK,both
2,SYN-S002,SYN-A,1,1,100.00,100.00,0,0.00,0.00,0.00,OK,OK,both
3,SYN-S003,SYN-B,2,1,300.00,150.00,-1,-150.00,0.00,-150.00,DE MENOS,PENDIENTE,both
4,SYN-S004,SYN-A,2,3,200.00,300.00,1,100.00,100.00,0.00,DE MÁS,JUSTIFICADO,both
5,SYN-S006,SYN-B,1,0,80.00,0.00,-1,-80.00,0.00,-80.00,DE MENOS,PENDIENTE,left_only
6,SYN-S007,SYN-A,0,1,0.00,90.00,1,90.00,90.00,0.00,DE MÁS,JUSTIFICADO,right_only


,raw_status,review_status,cases,students,difference_lines,difference_cents,justified_cents,pending_cents
0,DE MENOS,PENDIENTE,2,2,-2,-23000,0,-23000
1,DE MÁS,JUSTIFICADO,2,2,2,19000,19000,0
2,OK,OK,3,2,0,0,0,0


## 5. Evidence is consumed, not assumed / Evidencia consumida, no supuesta

The allocation ledger preserves the evidence ID, rule, direction and consumed capacities. Original difference = signed justification + pending. A partial explanation leaves the case open.

In [6]:
display(result.allocations)
for measure in ["lines", "cents"]:
    assert (result.detail[f"difference_{measure}"] ==
            result.detail[f"justified_{measure}"] + result.detail[f"pending_{measure}"]).all()
display(result.detail.loc[result.detail["review_status"].eq("PENDIENTE"),
                          [*KEYS, "pending_lines", "pending_cents", "raw_status"]])

,student_ref,program_ref,evidence_id,rule,direction,lines,cents
0,SYN-S004,SYN-A,SYNTHETIC_MANUAL_ADJUSTMENT:SYN-B08,SYNTHETIC_MANUAL_ADJUSTMENT,1,1,10000
1,SYN-S007,SYN-A,SYNTHETIC_MANUAL_ADJUSTMENT:SYN-B09,SYNTHETIC_MANUAL_ADJUSTMENT,1,1,9000


,student_ref,program_ref,pending_lines,pending_cents,raw_status
3,SYN-S003,SYN-B,-1,-15000,DE MENOS
5,SYN-S006,SYN-B,-1,-8000,DE MENOS


## 6. A failure totals alone would miss / Lo que el total oculta

Both sources contain 9 lines, but 890.00 actual − 930.00 expected = −40.00. Positive justified adjustments total 190.00; unresolved shortages total −230.00. Thus −40.00 = +190.00 − 230.00. Equal line totals do not make the individual keys correct.

Ambas fuentes contienen 9 líneas, pero 890,00 reales − 930,00 esperados = −40,00. Los ajustes positivos justificados suman 190,00 y los faltantes pendientes −230,00. Por eso −40,00 = +190,00 − 230,00. Igual cantidad total no implica claves correctas.

In [7]:
assert result.kpis["expected_lines"] == result.kpis["actual_lines"] == 9
assert result.kpis["mother_difference_cents"] == -4000
assert result.kpis["pending_absolute_cents"] == 23000
# Count-only discrepancy: same money, two expected lines vs one billed line.
count_expected = pd.DataFrame([["SYN-DEMO", "SYN-A", 2, 10000]],
                              columns=[*KEYS, "lines", "cents"])
count_actual = count_expected.assign(lines=1)
count_only = reconcile(count_expected, count_actual)
display(count_only.detail[[*KEYS, "difference_lines", "difference_cents", "review_status"]])
assert count_only.kpis["pending_cases"] == 1

,student_ref,program_ref,difference_lines,difference_cents,review_status
0,SYN-DEMO,SYN-A,-1,0,PENDIENTE


## 7. Independent source-to-detail bridge / Puente independiente

Totals are calculated before joining. Unassigned amounts remain separate from keyed pending exposure. A complete bridge is not a declaration that all business exceptions are resolved.

In [8]:
bridge = pd.DataFrame([
    {"measure": m, "expected": result.kpis[f"expected_{m}"],
     "actual": result.kpis[f"actual_{m}"],
     "mother_difference": result.kpis[f"mother_difference_{m}"],
     "keyed_detail": result.kpis[f"detail_difference_{m}"],
     "unassigned": result.kpis[f"unassigned_{m}"]}
    for m in ["lines", "cents"]
])
display(bridge)
assert (bridge["mother_difference"] == bridge["keyed_detail"] + bridge["unassigned"]).all()
display(pd.DataFrame([{
    "pending_cases": result.kpis["pending_cases"],
    "pending_net_cents": result.kpis["pending_net_cents"],
    "pending_absolute_cents": result.kpis["pending_absolute_cents"],
    "unassigned_cents": result.kpis["unassigned_cents"],
    "control_clear": result.kpis["control_clear"]
}]))

,measure,expected,actual,mother_difference,keyed_detail,unassigned
0,lines,9,9,0,0,0
1,cents,93000,89000,-4000,-4000,0


,pending_cases,pending_net_cents,pending_absolute_cents,unassigned_cents,control_clear
0,2,-23000,23000,0,False


## 8. Pipeline trace / Trazabilidad por etapas

The stage ledger records counts and check categories, never connection data or source records. Run the CLI with --verbose to see the same metadata in logging.

In [9]:
display(pd.DataFrame(result.trace))
assert result.trace[-1]["stage"] == "totals_bridge"
assert result.trace[-1]["check"] == "PASS"

,stage,rows,check
0,scope.assigned,9,validated_keys_and_row_ids; excluded_rows=0
1,scope.billed,9,validated_keys_and_row_ids; excluded_rows=0
2,input_contract,18,integer_measures;critical_keys;nonnegative_counts
3,aggregate.expected,6,unique_student_program
4,aggregate.actual,6,unique_student_program
5,join.outer,7,one_to_one;both_universes_preserved
6,classify,7,original_differences_preserved
7,allocate_evidence,2,unique_evidence;capacity_capped;direction_checked
8,totals_bridge,7,PASS


## Review outputs and limits / Outputs y límites

Run `python -m financial_controls.demo --period 2030-09 --output artifacts` to export detail, summary, allocations, unassigned records, trace, KPIs and HTML.

Amounts and counts are aggregate signals, not exact invoice matching. The synthetic flag does not prove a production adjustment was approved. Mixed line/amount directions are flagged. All examples use one fictitious currency.

Importes y cantidades son señales agregadas, no matching exacto de facturas. La bandera ficticia no prueba aprobación de un ajuste productivo. Las direcciones contradictorias se señalan. Se usa una moneda ficticia.

### Interview / Entrevista
- Why compare line count when amounts already reconcile? / ¿Por qué comparar cantidades si el importe concilia?
- How is partial evidence prevented from overjustifying a difference? / ¿Cómo se evita que evidencia parcial justifique de más?
- What additional keys would support invoice-level matching? / ¿Qué claves adicionales permitirían matching por factura?

No production processing, operational impact metric or confidential rule is reproduced.